# Demo 7: Task span vs code-progress span: which one holds the next sub-goal?

**Insight.** At the pivot, the LM reads its next sub-goal from the task span, not from the code it has written so
far. Transplanting the source's task span changes what the destination writes next; transplanting the code-progress
span barely does, even though its latents were computed while attending to the source's task list.

**Setup.**
\begin{align*}
p_\mathrm{src}: \mathrm{\underline{ab}c \rightarrow \mathrm{c}} \\
p_\mathrm{dst}: \mathrm{\underline{ab}d \rightarrow \mathrm{d}}
\end{align*}

As in Demo 5 (both prompts in Python here). By the pivot the LM has seen two things that could tell it what to write
next: the **task span** (the numbered list of sub-tasks) and the **code-progress span** (the code written so far, up to
the pivot). The code-progress span is token-identical in both prompts, but its latents are not: in SRC they were
computed while attending to SRC's task list. So we transplant each span from SRC into DST separately, a 2 × 2 grid:

| | progress from DST | progress from SRC |
|---|---|---|
| **task from DST** | clean DST (no-op) | progress only |
| **task from SRC** | task only | task + progress |

**Intervention.** Two ways to transplant a span. *Residual*: the span's input embeddings and every layer's outputs
are taken from SRC, and every other position before the pivot is pinned to clean DST. *KV through the goal heads*:
exactly Demo 5's, the 250 goal heads read SRC's keys and values over the span from the pivot on (values × 4). The
pivot itself always computes freely.

If DST now writes SRC's sub-task instead of its own, the transplanted span carried the next sub-goal.

**Paper.** App. J (Figures 27, 28).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (schematics from the lab talk):

![Task span or code progress: which one carries the goal? The 2×2 transfer grid (App. J).](figures/07_task_vs_progress_setup.png)

*Task span or code progress: which one carries the goal? The 2×2 transfer grid (App. J).*

![The (source-task, destination-progress) cell: residual patching of the task span, with the code progress restored from the destination.](figures/07_task_vs_progress_residual.png)

*The (source-task, destination-progress) cell: residual patching of the task span, with the code progress restored from the destination.*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# as in the paper's figures: source red, destination green, patched run purple
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    SRC_C, DST_C, PAT_C = 'salmon1', 'sea_green3', 'plum2'
    SRC_BG, DST_BG      = SRC_C, DST_C
    PIVOT_BG            = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    SRC_C, DST_C, PAT_C = '#cc0000', '#38761d', '#5b277d'
    SRC_BG, DST_BG      = '#f4cccc', '#d9ead3'
    PIVOT_BG            = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
DATATYPE, SRC_LANG, DST_LANG = 'int', 'python', 'python'
SHARED   = ('digit_sum', 'is_perfect_square')   # sub-tasks (1), (2): shared and prefilled
SRC_NEXT = 'count_set_bits'                     # the next sub-task in the SRC prompt
DST_NEXT = 'is_prime'                           # the next sub-task the DST prompt would do on its own
METHODS  = {'residual':          dict(method='residual', heads=None, v=1.0),     # every layer, every head
            'KV, goal 250, V x4': dict(method='kv', heads='goal250', v=4.0)}     # the goal heads' keys / values only
MAX_NEW  = 64

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct and the 250 goal heads.

In [3]:
import json, torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP
import src.nrgs.task_vs_progress as TP

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)

ctx = FUNC.setup(MODEL, device_map={'': 0})
mt = ctx.mt
goal = [tuple(h) for h in json.load(open(G250))]
headsets = {'goal250': goal}
print(f'goal heads: {len(goal)} of {mt.n_layer * mt.n_heads}')

2026-11-03 20:57:15 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 of 2560


## 1. The pair and its two spans
The two prompts as the model sees them: the task span highlighted, the code-progress span underlined, the pivot at
the end. The code-progress tokens are the same in both prompts; only their latents differ.

In [4]:
insts, _ = PP.sample_multi_instances(DATATYPE, SRC_LANG, DST_LANG, n_samples=32, seed=1123)
inst = next(i for i in insts if (tuple(p.id for p in i.shared), i.src_frontier.id, i.dst_frontier.id) == (SHARED, SRC_NEXT, DST_NEXT))
G = TP.geometry(inst, ctx=ctx)

def show(side, title, bg):
    ids = G[f'{side}_tok']['input_ids'][0].tolist()
    task, prog = (set(G['W'][s][0 if side == 'src' else 1]) for s in ('task', 'progress'))
    start = max(i for i, t in enumerate(ids) if t == mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(task))
    txt = Text()
    for i in range(start, len(ids)):
        pivot = i == len(ids) - 1                     # the pivot: the prompt's last token, ` =`
        st = (f'bold black on {PIVOT_BG}' if pivot else f'black on {bg}' if i in task
              else 'underline' if i in prog else 'dim')
        txt.append(mt.tokenizer.decode([ids[i]]), style=st)
    console.rule(title); console.print(txt)

show('src', 'SRC prompt (the donor)', SRC_BG)
show('dst', 'DST prompt (the run we patch)', DST_BG)

────────────────────────────────────────────────────────── SRC prompt (the donor) ──────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) the sum of the digits
(2) whether the number is a perfect square
(3) the number of set bits in the binary representation
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = sum(int(d) for d in str(abs(n)))
    b = n >= 0 and int(n**0.5)**2 == n
    c =

────────────────────────────────────────────────────── DST prompt (the run we patch) ───────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:




(1) the sum of the digits
(2) whether the number is a perfect square
(3) whether the number is prime
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = sum(int(d) for d in str(abs(n)))
    b = n >= 0 and int(n**0.5)**2 == n
    c =

## 2. Clean runs
Without any intervention, each prompt writes its own sub-task (3), and GoalScope at the pivot ` =` already says which
one it is about to write (as in Demo 2).

In [5]:
stub = lambda tok: mt.tokenizer.decode(tok['input_ids'][0].tolist()).rsplit('\n', 1)[-1]   # the prefilled `    c =`

def show_run(title, color, prompt, gen, readout=None, verdict=None):
    console.rule(f'[bold {color}]{title}[/]')
    if readout is not None:
        console.print(f'[bold]GoalScope at the pivot:[/] [italic]{escape(readout.strip())}[/]')
    console.print(f'[bold]continuation:[/]\n[grey50]{escape(stub(prompt))}[/][{color}]{escape(gen.rstrip())}[/]')
    if verdict: console.print(f'[bold]judge:[/] {verdict}')
    console.print()

with FUNC.console.capture():                               # silence the per-readout log lines
    clean = PP.clean_runs(dict(src_tok=G['src_tok'], dst_tok=G['dst_tok']), readout_heads=goal, max_new=MAX_NEW, ctx=ctx)
show_run('SRC run', SRC_C, G['src_tok'], clean['gen_src'], clean['ro_src'])
show_run('DST run', DST_C, G['dst_tok'], clean['gen_dst'], clean['ro_dst'])

───────────────────────────────────────────────────────────────── SRC run ──────────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm counting the number of 1s in the binary representation of an integer.

continuation:
    c = bin(n).count('1')
    return a, b, c
</code>

───────────────────────────────────────────────────────────────── DST run ──────────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the current number is prime.

continuation:
    c = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    return a, b, c
</code>

## 3. The 2 × 2 grid, twice
For each method, the three patched cells of the grid (the fourth is the clean DST run). For each patched run we show
the generation and the verdict of an LM judge (Claude): does the code's first computation implement SRC's sub-task (a
**hit**) or DST's own? A compact 2 × 2 summary follows each method.

In [6]:
SPANS = {('src', 'dst'): 'task', ('dst', 'src'): 'progress', ('src', 'src'): 'task+progress'}   # (task from, progress from)
lab = {'src': f'[bold {SRC_C}]c (SRC)[/]', 'dst': f'[bold {DST_C}]d (DST)[/]'}
console.print(f'[bold {SRC_C}]c[/] = {escape(inst.expected_next_desc)}   |   [bold {DST_C}]d[/] = {escape(inst.dst_next_desc)}\n')
for mname, m in METHODS.items():
    cells = {('dst', 'dst'): lab['dst']}
    for (ts, ps), span in SPANS.items():
        stack = TP.make_stack(G, m['method'], span, m['heads'], m['v'], headsets=headsets, ctx=ctx)
        gen = TP.run_patched(G, stack, max_new=MAX_NEW, ctx=ctx)
        causal = TP.judge(G, clean, gen)
        cells[ts, ps] = lab.get(causal['label'], causal['label'])
        verdict = f"code implements {cells[ts, ps]} ({'[bold]HIT[/]' if causal['triggered'] else '[dim]miss[/]'})"
        show_run(f'{mname}: task span from {ts.upper()}, progress span from {ps.upper()}', PAT_C, G['dst_tok'], gen,
                 verdict=verdict)
    t = Table(title=f'{mname}: what DST writes next')
    t.add_column(''); t.add_column('progress from DST'); t.add_column('progress from SRC')
    for ts in ('dst', 'src'):
        t.add_row(f'task from {ts.upper()}', cells[ts, 'dst'], cells[ts, 'src'])
    console.print(t); console.print()

c = the number of set bits in the binary representation   |   d = whether the number is prime

─────────────────────────────────────────── residual: task span from SRC, progress span from DST ───────────────────────────────────────────

continuation:
    c = bin(n).count('1') if n >= 0 else bin(-n).count('1')
    return a, b, c
</code>

judge: code implements c (SRC) (HIT)

─────────────────────────────────────────── residual: task span from DST, progress span from SRC ───────────────────────────────────────────

continuation:
    c = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
    return a, b, c
</code>

... (11 outputs left out)

────────────────────────────────────── KV, goal 250, V x4: task span from SRC, progress span from SRC ──────────────────────────────────────

continuation:
    c = bin(n).count('1') if n >= 0 else bin(n & -n).count('1')
    return a, b, c
</code>

judge: code implements c (SRC) (HIT)

        KV, goal 250, V x4: what DST writes next         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │ d (DST)           │ d (DST)           │
│ task from SRC │ c (SRC)           │ c (SRC)           │
└───────────────┴───────────────────┴───────────────────┘

## What to expect
<img src="figures/07_task_vs_progress_results.png" width="300" alt="Residual transfer of the task and code-progress spans, 2 x 2 grid (Figure 27, right, in the paper)">

Transplanting the task span should make DST write SRC's sub-task; transplanting the code-progress span alone should
leave DST on its own, and task + progress should behave like task alone. Averaged over 4 datatypes (32 instances
each), the residual transfer changes the code 89% of the time from the task span, 6% from the progress span and 98%
from both. Through the goal heads' keys and values alone (V × 4), the task span changes it 97% of the time and the
progress span 26%. See App. J (Figures 27, 28) in the paper.